In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression
import json

C:\Users\atliu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading Embeddings

In [2]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [4]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [5]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [6]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-03-11 17:02:00,289 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 17:02:58,468 - BERTopic - Dimensionality - Completed ✓
2026-03-11 17:02:58,471 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 17:03:07,827 - BERTopic - Cluster - Completed ✓
2026-03-11 17:03:07,854 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 17:03:25,826 - BERTopic - Representation - Completed ✓
2026-03-11 17:04:33,904 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 17:05:12,614 - BERTopic - Dimensionality - Completed ✓
2026-03-11 17:05:12,617 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 17:05:21,267 - BERTopic - Cluster - Completed ✓
2026-03-11 17:05:21,289 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 17:05:40,809 - BERTopic - Representation - Completed ✓
2026-03-11 17:06:16,937 - BERTopic - D

[(20, np.float64(0.4432309510898519), 708), (50, np.float64(0.469995752591413), 372), (100, np.float64(0.4984407949733464), 203)]


In [7]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.443231,708
1,50,0.469996,372
2,100,0.498441,203


In [8]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-03-11 17:07:52,389 - BERTopic - Topic reduction - Reducing number of topics
2026-03-11 17:07:52,494 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 17:08:12,214 - BERTopic - Representation - Completed ✓
2026-03-11 17:08:12,239 - BERTopic - Topic reduction - Reduced number of topics from 204 to 100
2026-03-11 17:08:15,570 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-11 17:08:15,972 - BERTopic - Dimensionality - Completed ✓
2026-03-11 17:08:15,973 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-11 17:08:34,088 - BERTopic - Cluster - Completed ✓


In [9]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [10]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [11]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [12]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
84    0.290696
41    0.090430
19    0.065883
44    0.061927
83    0.053476
79    0.047935
39    0.045902
3     0.032910
37    0.026620
36    0.025364
dtype: float64

📉 Declining topics:
40   -0.122298
96   -0.158126
95   -0.180363
87   -0.247353
66   -0.259540
80   -0.271171
88   -0.277157
76   -0.290427
71   -0.322944
49   -0.332887
dtype: float64


In [13]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
79    0.069411
19    0.066726
3     0.059828
83    0.044891
41    0.043703
78    0.025507
53    0.022026
37    0.021363
72    0.018967
8     0.018329
dtype: float64

📉 Declining topics:
59   -0.059745
66   -0.071180
40   -0.071434
91   -0.084672
97   -0.087825
95   -0.089085
71   -0.089830
76   -0.089870
49   -0.120194
70   -0.167120
dtype: float64


In [14]:
len(log_trend_scores_12), len(log_trend_scores_24)

(87, 92)

#  Visualization

In [15]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [16]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [17]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [18]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [19]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [20]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [21]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [22]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [23]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
0,9578,0_policy_reinforcement_learning_rl,"[policy, reinforcement, learning, rl, robot, t...",[Policy-Driven World Model Adaptation for Robu...,Reinforcement learning rl,-0.004541,-0.004038,📉 Declining,-0.000503,-0.041630,33.029995
1,7400,1_speech_audio_asr_recognition,"[speech, audio, asr, recognition, music, speak...",[On the Effect of Purely Synthetic Training Da...,Audio asr recognition,-0.000106,-0.000947,📉 Declining,0.000841,-0.000944,33.483000
2,5206,2_visual_multimodal_video_image,"[visual, multimodal, video, image, reasoning, ...",[What's in the Image? A Deep-Dive into the Vis...,Multimodal video image,-0.007254,0.003703,📈 Strong but Slowing,-0.010957,-0.062082,32.802277
3,4961,3_reasoning_cot_mathematical_llms,"[reasoning, cot, mathematical, llms, models, o...",[Question Decomposition Improves the Faithfuln...,Cot mathematical llms,0.032910,0.059828,🚀 Strong & Accelerating,-0.026918,0.280049,36.611617
4,4423,4_retrieval_rag_entity_the,"[retrieval, rag, entity, the, to, relation, on...",[Know3-RAG: A Knowledge-aware RAG Framework wi...,Rag entity the,-0.040196,-0.032052,📉 Declining,-0.008143,-0.337434,29.736461


In [24]:
topics_df["label"].nunique()

93

In [25]:
def get_topic_words(topic_id, top_n=10, model=reduced_model):
    
    words = model.get_topic(topic_id)
    
    words = words[:top_n]
    
    terms = [w[0] for w in words]
    scores = [w[1] for w in words]
    
    return terms, scores

In [26]:
topic_words = {}
for topic_id in topics_df.index:
    terms, scores = get_topic_words(topic_id)
    topic_words[topic_id] = (terms, scores)  

In [27]:
with open('topic_words.json', 'w') as f:
    json.dump(topic_words, f)

In [28]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [29]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [30]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [31]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [32]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [33]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [34]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [35]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [36]:
top10_impact.to_csv("top10_impact_topics.csv")

In [40]:
rep_docs = reduced_model.get_representative_docs()

top_docs = 5
rows = []

for topic_id, docs in rep_docs.items():
    for rant, text in enumerate(docs[:top_docs], start=1):
        rows.append({
            "topic_id": topic_id,
            "rank": rant,
            "text": text[:200]
        })
rep_docs_df = pd.DataFrame(rows)
rep_docs_df.to_csv("representative_docs.csv", index=False)

In [58]:
rep_docs[0]

['Policy-Driven World Model Adaptation for Robust Offline Model-based Reinforcement Learning Offline reinforcement learning (RL) offers a powerful paradigm for data-driven control. Compared to model-free approaches, offline model-based RL (MBRL) explicitly learns a world model from a static dataset and uses it as a surrogate simulator, improving data efficiency and enabling potential generalization beyond the dataset support. However, most existing offline MBRL methods follow a two-stage training procedure: first learning a world model by maximizing the likelihood of the observed transitions, then optimizing a policy to maximize its expected return under the learned model. This objective mismatch results in a world model that is not necessarily optimized for effective policy learning. Moreover, we observe that policies learned via offline MBRL often lack robustness during deployment, and small adversarial noise in the environment can lead to significant performance degradation. To addr

In [47]:
rep_docs[0]

['Policy-Driven World Model Adaptation for Robust Offline Model-based Reinforcement Learning Offline reinforcement learning (RL) offers a powerful paradigm for data-driven control. Compared to model-free approaches, offline model-based RL (MBRL) explicitly learns a world model from a static dataset and uses it as a surrogate simulator, improving data efficiency and enabling potential generalization beyond the dataset support. However, most existing offline MBRL methods follow a two-stage training procedure: first learning a world model by maximizing the likelihood of the observed transitions, then optimizing a policy to maximize its expected return under the learned model. This objective mismatch results in a world model that is not necessarily optimized for effective policy learning. Moreover, we observe that policies learned via offline MBRL often lack robustness during deployment, and small adversarial noise in the environment can lead to significant performance degradation. To addr

In [46]:
docs

['When Generative AI Meets Extended Reality: Enabling Scalable and Natural Interactions Extended Reality (XR), including virtual, augmented, and mixed reality, provides immersive and interactive experiences across diverse applications, from VR-based education to AR-based assistance and MR-based training. However, widespread XR adoption remains limited due to two key challenges: 1) the high cost and complexity of authoring 3D content, especially for large-scale environments or complex interactions; and 2) the steep learning curve associated with non-intuitive interaction methods like handheld controllers or scripted gestures. Generative AI (GenAI) presents a promising solution by enabling intuitive, language-driven interaction and automating content generation. Leveraging vision-language models and diffusion-based generation, GenAI can interpret ambiguous instructions, understand physical scenes, and generate or manipulate 3D content, significantly lowering barriers to XR adoption. This

In [44]:
pd.read_csv("representative_docs.csv").head()

,topic_id,rank,text
0,-1,1,LLMs Have Rhythm: Fingerprinting Large Languag...
1,-1,2,Evolution without Large Models: Training Langu...
2,-1,3,Data Distillation for Text Classification Deep...
3,0,1,Policy-Driven World Model Adaptation for Robus...
4,0,2,Scenario-Based Hierarchical Reinforcement Lear...


In [51]:
embeddings.shape

(179699, 384)

In [54]:
raw_data = pd.read_csv("arxiv_abstracts_final.csv").drop_duplicates()

In [55]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 179700 entries, 0 to 182968
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   paper_id    179700 non-null  object
 1   title       179700 non-null  object
 2   abstract    179700 non-null  object
 3   published   179700 non-null  object
 4   categories  179700 non-null  object
dtypes: object(5)
memory usage: 8.2+ MB
